In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("fetal health.csv.zip")

print("Shape (rows, columns):", df.shape)
display(df.head())

Shape (rows, columns): (2126, 22)


,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency,fetal_health
0,120.0,0.000,0.0,0.000,0.000,0.0,0.0,73.0,0.5,43.0,...,62.0,126.0,2.0,0.0,120.0,137.0,121.0,73.0,1.0,2.0
1,132.0,0.006,0.0,0.006,0.003,0.0,0.0,17.0,2.1,0.0,...,68.0,198.0,6.0,1.0,141.0,136.0,140.0,12.0,0.0,1.0
2,133.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.1,0.0,...,68.0,198.0,5.0,1.0,141.0,135.0,138.0,13.0,0.0,1.0
3,134.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,11.0,0.0,137.0,134.0,137.0,13.0,1.0,1.0
4,132.0,0.007,0.0,0.008,0.000,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,9.0,0.0,137.0,136.0,138.0,11.0,1.0,1.0


In [2]:
print("Missing values:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("Duplicate rows:", df.duplicated().sum())

df = df.drop_duplicates().reset_index(drop=True)

target_col = "fetal_health"
print("\nTarget value counts:\n", df[target_col].value_counts())
print("\nColumns:\n", list(df.columns))

Missing values:
 Series([], dtype: int64)
Duplicate rows: 13

Target value counts:
 fetal_health
1.0    1646
2.0     292
3.0     175
Name: count, dtype: int64

Columns:
 ['baseline value', 'accelerations', 'fetal_movement', 'uterine_contractions', 'light_decelerations', 'severe_decelerations', 'prolongued_decelerations', 'abnormal_short_term_variability', 'mean_value_of_short_term_variability', 'percentage_of_time_with_abnormal_long_term_variability', 'mean_value_of_long_term_variability', 'histogram_width', 'histogram_min', 'histogram_max', 'histogram_number_of_peaks', 'histogram_number_of_zeroes', 'histogram_mode', 'histogram_mean', 'histogram_median', 'histogram_variance', 'histogram_tendency', 'fetal_health']


In [3]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (1690, 21)
X_test shape: (423, 21)


In [4]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200, random_state=42, class_weight="balanced"
)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.9456264775413712

Confusion Matrix:
 [[328   2   0]
 [ 17  40   1]
 [  1   2  32]]

Classification Report:
               precision    recall  f1-score   support

         1.0       0.95      0.99      0.97       330
         2.0       0.91      0.69      0.78        58
         3.0       0.97      0.91      0.94        35

    accuracy                           0.95       423
   macro avg       0.94      0.87      0.90       423
weighted avg       0.94      0.95      0.94       423



In [5]:
import joblib

joblib.dump(rf_model, "fetal_rf_model.pkl")
joblib.dump(list(X.columns), "feature_columns.pkl")
print("Saved fetal_rf_model.pkl and feature_columns.pkl")

Saved fetal_rf_model.pkl and feature_columns.pkl


In [6]:
import joblib
import numpy as np
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Fetal Health Classifier", layout="wide")
st.title("Fetal Health Classifier (Random Forest)")

model = joblib.load("fetal_rf_model.pkl")
feature_cols = joblib.load("feature_columns.pkl")
df_ref = pd.read_csv("fetal health.csv.zip").drop(columns=["fetal_health"])

st.subheader("Enter Feature Values")
input_data = {}

cols = st.columns(3)
for idx, col in enumerate(feature_cols):
  default_val = float(df_ref[col].median())
  min_val = float(df_ref[col].min())
  max_val = float(df_ref[col].max())
  with cols[idx % 3]:
    input_data[col] = st.number_input(
        label=col,
        min_value=min_val,
        max_value=max_val,
        value=default_val,
    )

if st.button("Predict Fetal Health"):
  input_df = pd.DataFrame([input_data])[feature_cols]
  pred = int(model.predict(input_df)[0])
  probs = model.predict_proba(input_df)[0]

  label_map = {1: "Normal", 2: "Suspect", 3: "Pathological"}
  st.success(f"Predicted Class: {pred} ({label_map.get(pred, 'Unknown')})")

  prob_df = pd.DataFrame(
      {"Class": [label_map.get(int(c), str(c)) for c in model.classes_],
       "Probability": np.round(probs, 4)}
  )
  st.dataframe(prob_df, use_container_width=True)

2026-11-22 09:04:59.030 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-22 09:04:59.031 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-22 09:04:59.247 
  command:

    streamlit run /opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-11-22 09:04:59.248 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-22 09:04:59.248 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-22 09:04:59.310 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-22 09:04:59.311 Thread 'MainThread': missing ScriptRunContext! This warning

In [8]:
!streamlit run app.py

Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: app.py


In [11]:
%%writefile app.py
import joblib
import numpy as np
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Fetal Health Classifier", layout="wide")
st.title("Fetal Health Classifier (Random Forest)")

model = joblib.load("fetal_rf_model.pkl")
feature_cols = joblib.load("feature_columns.pkl")
df_ref = pd.read_csv("fetal health.csv.zip").drop(columns=["fetal_health"])

st.subheader("Enter Feature Values")
input_data = {}

cols = st.columns(3)
for idx, col in enumerate(feature_cols):
  default_val = float(df_ref[col].median())
  min_val = float(df_ref[col].min())
  max_val = float(df_ref[col].max())
  with cols[idx % 3]:
    input_data[col] = st.number_input(
        label=col,
        min_value=min_val,
        max_value=max_val,
        value=default_val,
    )

if st.button("Predict Fetal Health"):
  input_df = pd.DataFrame([input_data])[feature_cols]
  pred = int(model.predict(input_df)[0])
  probs = model.predict_proba(input_df)[0]

  label_map = {1: "Normal", 2: "Suspect", 3: "Pathological"}
  st.success(f"Predicted Class: {pred} ({label_map.get(pred, 'Unknown')})")

  prob_df = pd.DataFrame(
      {"Class": [label_map.get(int(c), str(c)) for c in model.classes_],
       "Probability": np.round(probs, 4)}
  )
  st.dataframe(prob_df, use_container_width=True)

Writing app.py


In [12]:
!ls -ls

total 4844
   0 lrwxrwxrwx 1 bff74a6b-e34a-434d-a774-51ef219eb102 registered_users      30 Nov 22 06:34  anaconda_projects -> /var/www/filebrowser/.projects
   4 -rw-rw-r-- 1 bff74a6b-e34a-434d-a774-51ef219eb102 registered_users    1283 Nov 22 09:13  app.py
   4 -rw-rw-r-- 1 bff74a6b-e34a-434d-a774-51ef219eb102 registered_users     531 Nov 22 09:04  feature_columns.pkl
  48 -rw-rw-r-- 1 bff74a6b-e34a-434d-a774-51ef219eb102 registered_users   46932 Nov 22 08:55 'fetal health.csv.zip'
  44 -rw-rw-r-- 1 bff74a6b-e34a-434d-a774-51ef219eb102 registered_users   42107 Nov 22 09:14  fetal.ipynb
4744 -rw-rw-r-- 1 bff74a6b-e34a-434d-a774-51ef219eb102 registered_users 4854657 Nov 22 09:04  fetal_rf_model.pkl


In [ ]:
!streamlit run app.py




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://10.0.0.165:8501
  External URL: http://10.0.0.165:8501



In [ ]:
%%writefile requirements.txt
streamlit
pandas
numpy
scikit-learn
joblib